# OSM vs ENTSO-E GridKit

## Libraries

In [ ]:
# loading required libraries
import pypsa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cartopy import crs as ccrs
import xarray as xr
import folium
from folium import plugins
import geopandas as gpd
from branca.element import Figure
import branca.colormap as cm
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from shapely import box, normalize
from shapely.geometry import Polygon, LineString, Point
from shapely.wkt import loads

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

## Part I: Setup and visualisation

In [ ]:
network_entsoe = pypsa.Network("networks/entsoe_europe/base.nc")
network_osm = pypsa.Network("networks/osm_europe/base.nc")

print(network_entsoe)
print(network_osm)

In [ ]:
# Quick plots of the networks
fig, axes = plt.subplots(1, 2, figsize=(10, 5), subplot_kw={"projection": ccrs.EqualEarth()})

network_entsoe.plot(ax=axes[0], title="ENTSO-E\n", bus_sizes=0, line_widths=0.3, link_widths=0.3)
network_osm.plot(ax=axes[1], title="OSM\n", bus_sizes=0, line_widths=0.3, link_widths=0.3)

# Show the plot
plt.show()

In [ ]:
# Define function to create lines using the start bus and end bus coordinates.
# This method yields point-to-point straight lines, returned as list of 
# shapely.geometry.linestring.LineString
def lineGeometry(network):
    geometry = [LineString(
        [(network.buses.loc[row.bus0]["x"], network.buses.loc[row.bus0]["y"]), 
         (network.buses.loc[row.bus1]["x"], network.buses.loc[row.bus1]["y"])]
         ) \
            for idx, row in network.lines.iterrows()]

    return geometry

def linkGeometry(network):
    geometry = [LineString(
        [(network.buses.loc[row.bus0]["x"], network.buses.loc[row.bus0]["y"]), 
         (network.buses.loc[row.bus1]["x"], network.buses.loc[row.bus1]["y"])]
         ) \
            for idx, row in network.links.iterrows()]

    return geometry

In [ ]:
# Use shapely.wkt.loads to convert the string representation of the geometry to
# a shapely object. This is only available for OSM data from pypsa-earth.
network_osm.lines["geometry"] = network_osm.lines["geometry"].apply(loads)

In [ ]:
# Create a GeoDataFrame for the ENTSO-E network using the lineGeometry function
gdf_entsoe_lines = gpd.GeoDataFrame(network_entsoe.lines, geometry = lineGeometry(network_entsoe), crs = "EPSG:4326")
gdf_entsoe_links = gpd.GeoDataFrame(network_entsoe.links, geometry = linkGeometry(network_entsoe), crs = "EPSG:4326")

In [ ]:
# Create a GeoDataFrame for the OSM network using the the lineGeometry function
gdf_osm_lines = gpd.GeoDataFrame(network_osm.lines, geometry = lineGeometry(network_osm), crs = "EPSG:4326")
gdf_osm_links = gpd.GeoDataFrame(network_osm.links, geometry = linkGeometry(network_osm), crs = "EPSG:4326")

# Create a GeoDataFrame for the OSM network using accurate LineString geoms
gdf_osm_lines_acc = gpd.GeoDataFrame(network_osm.lines, geometry = network_osm.lines["geometry"], crs = "EPSG:4326")

In [ ]:
list_v_nom = np.concatenate(
    [gdf_entsoe_lines.v_nom.unique(), 
    gdf_osm_lines.v_nom.unique()]
    )

list_v_nom = np.unique(list_v_nom)
list_v_nom

In [ ]:
colormap = cm.LinearColormap(
    ["orange", "purple", "black"], 
    vmin = gdf_entsoe_lines["v_nom"].min(),
    vmax = gdf_entsoe_lines["v_nom"].max()
    )

colormap = colormap.to_step(index = list_v_nom)
colormap.caption = "Voltage level in kV"
colormap

In [ ]:
# Substations
buses_xy_entsoe = gpd.points_from_xy(network_entsoe.buses.x, network_entsoe.buses.y)
buses_xy_osm = gpd.points_from_xy(network_osm.buses.x, network_osm.buses.y)

gdf_entsoe_buses = gpd.GeoDataFrame(geometry = buses_xy_entsoe, crs = "EPSG:4326")
gdf_osm_buses = gpd.GeoDataFrame(geometry = buses_xy_osm, crs="EPSG:4326")

In [ ]:
# Define tooltip that is shown when hovering over the lines
def tooltip_lines():
    tooltip = folium.GeoJsonTooltip(
        fields=["Line", "bus0", "bus1", "v_nom", "length"],
        aliases=["Line:", "Bus 0:", "Bus 1:", "Voltage (kV):", "Length (km):"],
        labels = True,
        localize = True,
        sticky = False
    )

    return tooltip

def tooltip_links():
    tooltip = folium.GeoJsonTooltip(
        fields=["Link", "bus0", "bus1", "p_nom", "length"],
        aliases=["Line:", "Bus 0:", "Bus 1:", "P (nom.) (MW):", "Length (km):"],
        labels = True,
        localize = True,
        sticky = False
    )

    return tooltip

# Determine the center of the map
loc_x = np.array([network_entsoe.buses.x.min(), network_entsoe.buses.x.max()]).mean()
loc_y = np.array([network_entsoe.buses.y.min(), network_entsoe.buses.y.max()]).mean()

# Create the folium map using CartoDB positron tiles
m = folium.plugins.DualMap(location=(loc_y, loc_x), zoom_start=7, tiles = "CartoDB positron")

m1_title = "ENTSO-E network"
m2_title = "OSM network"

m.m1.get_root().html.add_child(
    folium.Element(f"<h1 style='position:absolute;z-index:100000;left:20vw'>{m1_title}</h1>")
    )

m.m2.get_root().html.add_child(
    folium.Element(f"<h1 style='position:absolute;z-index:100000;right:20vw'>{m2_title}</h1>")
    )

# Add the colormap (legend) 
colormap.add_to(m.m2)

# Add the feature groups to the map
fg_entsoe_buses = folium.FeatureGroup(name = "Buses ENTSO-E").add_to(m.m1)
fg_osm_buses = folium.FeatureGroup(name = "Buses OSM").add_to(m.m2)

fg_entsoe_lines = folium.FeatureGroup(name = "Lines ENTSO-E").add_to(m.m1)
fg_osm_lines = folium.FeatureGroup(name = "Lines OSM", show = False).add_to(m.m2)
fg_osm_lines_acc = folium.FeatureGroup(name = "Lines OSM (acc.)").add_to(m.m2)

fg_entsoe_links = folium.FeatureGroup(name = "Links ENTSO-E").add_to(m.m1)
fg_osm_links = folium.FeatureGroup(name = "Links OSM").add_to(m.m2)

# Define the callback function for the FastMarkerCluster
callback = (
    'function (row) {'
    'var circle = L.circle(new L.LatLng(row[0], row[1]), {color: "red",  radius: 5000});'
    'return circle};'
    )

# Add the layer control to the map
folium.LayerControl(collapsed=False).add_to(m)

# Add the buses to the feature groups
folium.plugins.FastMarkerCluster(
    network_entsoe.buses[["y", "x"]].values.tolist(), 
    callback = callback, disableClusteringAtZoom = 6,
    ) \
        .add_to(fg_entsoe_buses)

folium.plugins.FastMarkerCluster(
    network_osm.buses[["y", "x"]].values.tolist(), 
    callback = callback, disableClusteringAtZoom = 6,
    fill_opacity = 0.1
    ) \
        .add_to(fg_osm_buses)

# Add the lines to the feature groups
folium.GeoJson(
    gdf_entsoe_lines.reset_index(), 
    style_function=lambda feature: {
        "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_lines(),
    ) \
        .add_to(fg_entsoe_lines)

folium.GeoJson(
    gdf_osm_lines.reset_index(), 
    style_function=lambda feature: {
        "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_lines(),
    ) \
        .add_to(fg_osm_lines)

folium.GeoJson(
    gdf_osm_lines_acc.reset_index(), 
    style_function=lambda feature: {
        "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_lines(),
    ) \
        .add_to(fg_osm_lines_acc)

# Add the links to the feature groups
folium.GeoJson(
    gdf_entsoe_links.reset_index(), 
    style_function=lambda feature: {
        # "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_links(),
    ) \
        .add_to(fg_entsoe_links)

folium.GeoJson(
    gdf_osm_links.reset_index(), 
    style_function=lambda feature: {
        # "color": colormap(feature["properties"]["v_nom"]),
        "weight": 2,
        "opacity": 0.8,
        },
    tooltip = tooltip_links(),
    ) \
        .add_to(fg_osm_links)

# Export the map to a self-contained .html file
m.save("map.html")

### Map

Lets explore the two networks side-by-side using leaflet/folium below.

In [ ]:
# Display interactive map
m

## Part II: Numerical analysis

### Preparation

In a next step, we apply different metrics to compare the quality of the two network datasets. To do this on a regional level, we use NUTS2 regions at the highest resolution available, provided in the EPSG:4326 geographic coordinate system (to be consistent with the provided networks). First, we download the latest official data provided by Eurostat and import it using GeoPandas.

In [ ]:
url_nuts2 = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_01M_2021_4326_LEVL_2.geojson"
url_nuts2 = "geojson/NUTS_RG_01M_2021_4326_LEVL_2.geojson"

gdf_nuts2 = gpd.read_file(url_nuts2).drop(columns=["NUTS_ID", "LEVL_CODE", "NUTS_NAME", "MOUNT_TYPE", "URBN_TYPE", "COAST_TYPE", "FID"])
gdf_nuts2.columns

In [ ]:
gdf_nuts2["type"] = "NUTS2"
gdf_nuts2.rename(columns = {"CNTR_CODE": "country_code", "id": "region_code", "NAME_LATN": "name"}, inplace = True)
gdf_nuts2.reindex(columns = ["region_code", "name", "country_code", "type", "geometry"])
gdf_nuts2.columns

# Rename Eurostat country code for "EL" to "GR"
gdf_nuts2["country_code"] = gdf_nuts2["country_code"].replace("EL", "GR")

# Rename region_code prefix "EL" to "GR" for consistency with updated country code for Greece
gdf_nuts2["region_code"] = gdf_nuts2["region_code"].str.replace("EL", "GR")
gdf_nuts2.columns

We see that shapes for BA and UA are missing. To obtain equivalent regional administration boundaries UA, we use the following.

In [ ]:
# BA
url_ba = "https://github.com/wmgeolab/geoBoundaries/raw/905b0ba/releaseData/gbOpen/BIH/ADM1/geoBoundaries-BIH-ADM1.geojson"
url_ba="geojson/geoBoundaries-BIH-ADM1.geojson"

gdf_ba = gpd.read_file(url_ba).drop(columns=["shapeID", "shapeGroup", "shapeType"])
gdf_ba.rename(columns = {"shapeName": "name", "shapeISO": "region_code"}, inplace = True)
gdf_ba["region_code"] = gdf_ba["region_code"].str.replace("-", "")
gdf_ba["country_code"], gdf_ba["type"] = ["BA", "ADM1"]
gdf_ba = gdf_ba.reindex(columns = ["region_code", "name", "country_code", "type", "geometry"])
gdf_ba.columns

In [ ]:
# UA
url_ua = "https://github.com/wmgeolab/geoBoundaries/raw/905b0ba/releaseData/gbOpen/UKR/ADM1/geoBoundaries-UKR-ADM1.geojson"
url_ua = "geojson/geoBoundaries-UKR-ADM1.geojson"
gdf_ua = gpd.read_file(url_ua).drop(columns=["shapeID", "shapeGroup", "shapeType"])
gdf_ua.rename(columns = {"shapeName": "name", "shapeISO": "region_code"}, inplace = True)
gdf_ua["region_code"] = gdf_ua["region_code"].str.replace("-", "")
gdf_ua["country_code"], gdf_ua["type"] = ["UA", "ADM1"]
gdf_ua = gdf_ua.reindex(columns = ["region_code", "name", "country_code", "type", "geometry"])
gdf_ua.columns

Now let's merge the two GeoDataFrames and remove not needed regions.

In [ ]:
gdf_regions = pd.concat([gdf_nuts2, gdf_ba, gdf_ua])
gdf_regions = gdf_regions[(gdf_regions["country_code"] != "IS") & (gdf_regions["country_code"] != "TR")]

# Set index of gdf_regions to region_code
gdf_regions.set_index("region_code", inplace = True)

In [ ]:
# Define function that returns gdf_metrics using the input gdf_regions and gdf_lines
def getMetrics(gdf_regions, gdf_lines):

    print("Obtain list of regions that contain lines.")

    # Obtain a sorted list of region codes that intersect with the lines
    regions_intersect = sorted(
          gpd.overlay(gdf_lines.reset_index(), gdf_regions.reset_index(), how = "intersection")["region_code"] \
            .unique()
          )
    
    print("Start processing:")
    
    gdf_metrics = gpd.GeoDataFrame(columns=["region_code", "Line", "geometry", "s_nom", "v_nom", "length"], geometry="geometry").set_index(["region_code", "Line"])
    
    # for idx_region, region in gdf_regions.iterrows():
    for idx_region in regions_intersect:
        gdf_lines_region = gpd.overlay(gdf_lines.reset_index(), gdf_regions.loc[[idx_region]].reset_index(), how = "intersection").set_index(["region_code", "Line"])
        
        # Calculate length using the estimate_utm_crs method --> returns the length in meters
        utm = gdf_lines_region.loc[[idx_region]].estimate_utm_crs(datum_name = "WGS 84")
        gdf_lines_region.loc[[idx_region], ["length_calc"]] = gdf_lines_region.loc[[idx_region]].to_crs(utm).length/1e3

        gdf_metrics = pd.concat([gdf_metrics, gdf_lines_region[["geometry", "s_nom", "v_nom", "length", "length_calc"]]])
        print("Region", idx_region, "processed.")

    print("Processing finished.")
    print("Calculate line volumes")
    gdf_metrics["line_volume"] = gdf_metrics["length"] * gdf_metrics["s_nom"]
    gdf_metrics["line_volume_calc"] = gdf_metrics["length_calc"] * gdf_metrics["s_nom"]
    
    print("Complete.")
    return gdf_metrics

In [ ]:
gdf_metrics_entsoe = getMetrics(gdf_regions, gdf_entsoe_lines)
gdf_metrics_entsoe.head()

In [ ]:
gdf_metrics_osm = getMetrics(gdf_regions, gdf_osm_lines_acc)
gdf_metrics_osm.head()

In [ ]:
# Group metrics entso e by region_code and v_nom:
gdf_metrics_entsoe_grouped = gdf_metrics_entsoe.groupby(["region_code"]).agg({"length": "sum", "length_calc": "sum", "line_volume": "sum", "line_volume_calc": "sum"})
gdf_metrics_entsoe_grouped.head()

In [ ]:
gdf_metrics_osm_grouped = gdf_metrics_osm.groupby(["region_code"]).agg({"length": "sum", "length_calc": "sum", "line_volume": "sum", "line_volume_calc": "sum"})
gdf_metrics_osm_grouped.head()

In [ ]:
fig = make_subplots(rows = 1, cols = 2, subplot_titles = ("Line volume", "Calculated line volume"),
shared_yaxes = "all")

fig.add_trace(
    go.Scatter(
        x = gdf_metrics_entsoe_grouped["line_volume"],
        y = gdf_metrics_osm_grouped["line_volume"],
        mode = "markers",
        marker = dict(color = "blue"),
        name = "Line volume (MVAkm)"
        ),
    row = 1, col = 1
    )

fig.add_trace(  
    go.Scatter(
        x = gdf_metrics_entsoe_grouped["line_volume_calc"],
        y = gdf_metrics_osm_grouped["line_volume_calc"],
        mode = "markers",
        marker = dict(color = "green"),
        name = "Calculated line volume (MVAkm)"
        ),
    row = 1, col = 2
    )


# Add labels 
fig.update_traces(
    hovertemplate = "<b>Region</b>: %{text}<br><b>ENTSO-E</b>: %{x} km<br><b>OSM-E</b>: %{y} km",
    text = gdf_metrics_osm_grouped.index
    )

fig.update_traces(
    hovertemplate = "<b>Region</b>: %{text}<br><b>ENTSO-E</b>: %{x} km<br><b>OSM-E</b>: %{y} km",
    text = gdf_metrics_osm_grouped.index
    )

# Add 1:1 line to both subplots
fig.add_trace(
    go.Scatter(
        x = [0, 18*1e6],
        y = [0, 18*1e6],
        mode = "lines",
        line = dict(color = "black", width = 1),
        showlegend = False
        ),
    row = 1, col = 1
    )

fig.add_trace(
    go.Scatter(
        x = [0, 18*1e6],
        y = [0, 18*1e6],
        mode = "lines",
        line = dict(color = "black", width = 1),
        showlegend = False
        ),
    row = 1, col = 2
    )

# keep x and y axis ratio equal in all three subplots
fig.update_xaxes(scaleanchor = "x", scaleratio = 1, row = 1, col = 1)
fig.update_xaxes(scaleanchor = "x", scaleratio = 1, row = 1, col = 2)

# make both subplots square 
fig.update_layout(
    title_text = "Comparison of line volume metrics",
    width = 1000, height = 500
    )

fig.show()

In [ ]:
# Create box plot using two subplots
# Left contains delta between line volumes OSM minus ENTSO-E
# Right contains delta between calculated line volumes OSM minus ENTSO-E

fig = make_subplots(rows = 1, cols = 2, subplot_titles = ("Line volume", "Calculated line volume"),
                    shared_yaxes = "all")

fig.add_trace(
    go.Box(
        y = gdf_metrics_osm_grouped["line_volume"]-gdf_metrics_entsoe_grouped["line_volume"],
        name = "Line volume (MVAkm)",
        marker = dict(color = "blue")
        ),
    row = 1, col = 1
    )

fig.add_trace(
    go.Box(
        y = gdf_metrics_osm_grouped["line_volume_calc"]-gdf_metrics_entsoe_grouped["line_volume_calc"],
        name = "Calculated line volume (MVAkm)",
        marker = dict(color = "green")
        ),
    row = 1, col = 2
    )

fig.update_layout(
    title_text = "Comparison of line volume metrics",
    width = 1000, height = 500
    )

fig.show()

In [ ]:
# fig_m2 = Figure(width = "50%", height = 600)
# m2 = gdf_osm_lines_acc.explore(color = "orange", name = "Lines")
# m2 = gdf_regions.explore(m = m2, name = "Administrative regions")

# folium.LayerControl(collapsed = False).add_to(m2)

# fig_m2.add_child(m2)
# m2